# Pandas Module

This document includes few tips for the `pandas` module.

**Be aware that also pandas is using views for slicing and some methods!**

In [ ]:
# Let's examine the Iris flower dataset together.

import seaborn as sns
import pandas as pd

df = sns.load_dataset('iris')

In [ ]:
# The different "shapes" of the Pandas data

# A DataFrame is the entire table — a two-dimensional structure with rows and columns.
print(f'DataFrame iris:\n {df}')


In [ ]:
# A Series is a single column (or a single row) of data. It is "one-dimensional".
# Technically, a Pandas Series is 1-dimensional because it only contains one column of "actual data."
# However, it always comes with an Index (the labels).
# Think of it as a specialized Python Dictionary where the Index is the Key and the Data is the Value.
sepal_length = df['sepal_length']
print(f'Series sepal_length type:\n {type(sepal_length)}')
print(f'Series sepal_length:\n {sepal_length.head(3)}')
print(30 * '-')

# Getting multiple series is also possible -> This creates again a Dataframe
sepal = df[['sepal_length', 'sepal_width']]
print(f'Series sepal type:\n {type(sepal)}')
print(f'Series sepal:\n {sepal.head(3)}')
print('Shape of multiple series:', sepal.shape)

In [ ]:
# When you use .groupby(), Pandas doesn't create a new table immediately.
# Instead, it creates a "hidden" map of which rows belong to which group.
# This is done via the Index of the Dataframe
grouped = df.groupby('species')
print(f'Type of grouped data: {type(grouped)}')
indcies_setosa = (grouped.groups)['setosa']
indcies_virginica = (grouped.groups)['virginica']
print(f'Grouped with species:\n {indcies_setosa}\n{indcies_virginica}')

In [ ]:
# Grouping by Multiple Columns: The MultiIndex
# When you pass a list of columns to .groupby(['col1', 'col2']), Pandas creates a hierarchical index (MultiIndex).
# (use penguins dataset, because 'iris' has only 1 non-numerical column)
penguins = sns.load_dataset('penguins')
penguins_grouped = penguins.groupby(['species', 'island'])
print(f'Type of grouped data: {type(penguins_grouped)}')
print(f'Keys of grouped data (Multiindex -> tuple): {penguins_grouped.groups.keys()}')
# -> all keys: dict_keys([('Adelie', 'Biscoe'), ('Adelie', 'Dream'), ('Adelie', 'Torgersen'), ('Chinstrap', 'Dream'), ('Gentoo', 'Biscoe')])
indcies_0 = (penguins_grouped.groups)[('Adelie', 'Biscoe')]
indcies_1 = (penguins_grouped.groups)[('Adelie', 'Dream')]
# ... 
print(f'Grouped with species:\n {indcies_0}\n{indcies_1}')
# Or get it back as a Dataframe
adelie_biscoe_rows = penguins_grouped.get_group(('Adelie', 'Biscoe'))
print(f'Grouped adelie_biscoe_rows Dataframe:\n {type(adelie_biscoe_rows)}\n{adelie_biscoe_rows.tail(5)}')

In [ ]:
# Whether .groupby() returns a DataFrame or a Series depends on your selection:
# df.groupby(['A', 'B']).mean() -> Returns a DataFrame.
# It calculates the mean for all numeric columns. Multiple columns = DataFrame.

# df.groupby(['A', 'B'])['C'].mean() -> Returns a Series.
# You specifically asked for column 'C' only and used the aggragation function 'mean'.
# One column of data = Series (even if the index has multiple levels!).
series_result = penguins_grouped['body_mass_g'].mean()
print(type(series_result))
# Here you see the Multiindex as a tuple of species and island and the data is the mean values of body_mass_g for each group.
print(series_result)
# In Pandas, even though a Series is "one-dimensional," it can have a .name attribute.
# When you run .reset_index(), Pandas uses this name to label the new column in the resulting DataFrame.
# See in Output of this cell: Name: body_mass_g, dtype: float64 and next cell the reset_index example.

In [ ]:
# As seen previously:
# When you calculate the mean of a specific column after grouping by two categories, you get a Series with a MultiIndex.
# sns.barplot() mmeds a dataframe with a x and y variable (both columns)
# Solution: reset_index "flatten" the data of a series -> It turns the Index levels back into regular columns.
plot_data = series_result.reset_index()
print(plot_data)
sns.barplot(data=plot_data, x='species', y='body_mass_g', hue='island')

In [ ]:
# Difference to unstack:
# unstack() takes the innermost index level (the one on the right) and pivots it into column headers.
# Heatmaps: If you want to use sns.heatmap(), your data must be in this 'wide' matrix format.
# Human Readability: It’s much easier for a human to compare islands side-by-side in a table.

matrix_df = series_result.unstack()
print(f'--- UNSTACKED (Wide Format) with type: {type(matrix_df)} ---')
print(matrix_df)
sns.heatmap(data=matrix_df, cmap='Reds')

# Pandas Cheat Sheet

import pandas as pd
import io

# Sample Data
csv_data = """Name,Dept,Salary,Bonus,Remote,Age
Alice,IT,95000,5000,False,NaN
Bob,Sales,70000,2000,True,NaN
Charlie,IT,88000,4000,False,45
Diana,HR,60000,1000,True,NaN
Eric,Sales,55000,1500,False,67
Fiona,IT,110000,10000,False,NaN
George,HR,75000,3000,True,23
"""

df = pd.read_csv(io.StringIO(csv_data))
print(df)

In [ ]:
import pandas as pd
import io

# Sample Data
csv_data = """Name,Dept,Salary,Bonus,Remote,Age
Alice,IT,95000,5000,False,NaN
Bob,Sales,70000,2000,True,NaN
Charlie,IT,88000,4000,False,45
Diana,HR,60000,1000,True,NaN
Eric,Sales,55000,1500,False,67
Fiona,IT,110000,10000,False,NaN
George,HR,75000,3000,True,23
"""

df = pd.read_csv(io.StringIO(csv_data))
print(df)

In [ ]:
# --- 1. GROUPBY & AGGREGATION ---
# Concept: Split the data into groups (by Dept), apply a function (mean), 
# and combine the results. 
# Note: This creates a "Grouped" object. To get a DataFrame back, we use .mean()
print(df.groupby('Dept').groups)
print(50 * '-')
print(df.groupby('Dept').get_group('IT'))
print(50 * '-')
avg_salary_per_dept = df.groupby('Dept')['Salary'].mean()
print(avg_salary_per_dept)

In [ ]:
# --- 2. RESET_INDEX ---
# Concept: After a GroupBy, the grouping column ('Dept') becomes the Index.
# reset_index() moves the index back into a regular column and creates a 
# standard 0, 1, 2... integer index. This is essential for further plotting or merging.
df_dept_stats = avg_salary_per_dept.reset_index()
print("--- After GroupBy & Reset Index ---\n", df_dept_stats)

In [ ]:
# --- 3. PIVOT TABLE ---
# Concept: A 2D-Aggregation. It summarizes data across two variables (Rows & Columns).
# index: What defines the rows (vertical)
# columns: What defines the columns (horizontal)
# values: The numeric data to summarize
# aggfunc: The math to apply (mean, sum, count, etc.)
salary_matrix = df.pivot_table(
    index='Dept', 
    columns='Remote', 
    values='Salary', 
    aggfunc='mean'
)
print("\n--- Pivot Table (Mean Salary by Dept/Remote) ---\n", salary_matrix)

In [ ]:
# --- 4. SORT_VALUES ---
# Concept: Reorders the rows based on one or more columns.
# ascending=False: Sorts from highest to lowest.
# na_position: Where to put NaN values ('last' or 'first').
sorted_df = df.sort_values(by='Salary', ascending=False)
print("\n--- Sorted by Salary (Descending) ---\n", sorted_df[['Name', 'Salary']])

In [ ]:
# --- 5. ILOC (Integer Location) ---
# Concept: Purely integer-location based indexing for selection by position.
# Syntax: df.iloc[row_index, column_index]
# Works like a 2D-array access. [0:3] means rows 0, 1, and 2.
print(df)
top_3_rows_first_2_cols = df.iloc[0:3, 0:2]
print("\n--- iloc Selection (First 3 rows, first 2 columns) ---\n", top_3_rows_first_2_cols)

In [ ]:
# --- 6. BOOLEAN FILTERING (Bonus Concept) ---
# Concept: Create a mask (True/False list) and apply it to the DataFrame.
high_earners = df[df['Salary'] > 80000]
print("\n--- High Earners (> 80k) ---\n", high_earners)

only_it = df[df['Dept'] == 'IT']
print("\n--- Only It ---\n", only_it)

In [ ]:
# --- 7. HANDLING MISSING DATA (Finding & Filling NaNs) ---
# Concept: Real-world data often has "holes" (NaN = Not a Number). 
# You must decide: Fill them (Imputation) or Delete them (Dropping).
# .isna().sum() gives you a count of missing values per column.
missing_count = df.isna().sum()
print(missing_count)

In [ ]:
# Example: Dropping rows where 'Age' is missing (if it's only 1-2 rows)
df_cleaned = df.dropna(subset=['Age'])
print(df_cleaned)
print(50 * '-')

# Example: Filling missing 'Age' with the median to avoid the "Mean-Peak"
df['Age'] = df['Age'].fillna(df['Age'].median())
print(df)

In [ ]:
# --- 8. MERGING & JOINING (Combining DataFrames) ---
# Concept: Like a SQL JOIN. Use this to combine two tables based on a common key.
# how='inner': Keeps only keys present in BOTH DataFrames.
# how='left': Keeps all rows from the left DF, adds data from the right where available.
# on='ID': The shared column name (foreign key).
extra_info = pd.DataFrame({'Dept': ['IT', 'Sales'], 'Floor': [3, 1]})
df_merged = pd.merge(df, extra_info, on='Dept', how='left')

print("\n--- Merged Frame ---\n", df_merged)

In [ ]:
# --- 9. STRING OPERATIONS (.str Accessor) ---
# Concept: Allows you to treat an entire column of strings as a single object 
# to perform operations like searching, slicing, or case conversion.
# Very useful for cleaning text data (Feature Engineering).
df['Dept_Upper'] = df['Dept'].str.upper()
df['Is_Manager'] = df['Name'].str.contains('Manager', case=False)

In [ ]:
# --- 10. METHOD CHAINING (Clean & Readable Logic) ---
# Concept: Instead of creating 10 temporary variables, you link commands 
# with dots. Wrapping the whole block in parentheses () allows for line breaks.
# This is the "Pro-Way" to write readable transformation pipelines.
final_report = (
    df.query("Salary > 50000")              # 1. Filter rows
      .groupby("Dept")["Salary"].mean()     # 2. Group and Calc Mean
      .reset_index()                        # 3. Flatten the index
      .sort_values("Salary", ascending=False) # 4. Sort results
)

print("\n--- Final Chained Report ---\n", final_report)

In [ ]:
# --- BONUS: MEMORY & INFO ---
# Concept: Check data types (Dtypes) and memory usage. 
# Crucial for C++ devs to see if an 'int' is stored as a 64-bit float unnecessarily.
print("\n--- Dataframe Info ---")
df.info()